# Tutorial 6.2: RC model identification and validation

This tutorial estimates a compact $1R1C$ controller model from noisy measurements generated by the independent reference plant. The model is evaluated on a later, untouched period.

## Learning outcomes

By the end of the tutorial, you should be able to:

1. align thermal inputs with the temperature transition they cause;
2. fit bounded coefficients using physical knowledge;
3. recover effective $R$, $C$ and $	au$ values; and
4. distinguish one-step validation from free rollout.

## Indicative schedule

- data inspection and alignment: 15 minutes;
- least-squares fitting: 25 minutes;
- parameter recovery: 15 minutes;
- validation and residuals: 30 minutes;
- discussion: 5 minutes.

The optional occupied-house extension adds approximately 30 minutes and can also be completed independently after class.

## 1. Set-up and data

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import lsq_linear

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    raise FileNotFoundError("Run this notebook from the repository root.")

plt.rcParams.update({"figure.figsize": (10, 4), "figure.dpi": 110})

In [ ]:
data = pd.read_csv(
    ROOT / "data" / "reference_building" / "identification_data.csv",
    index_col="timestamp",
    parse_dates=True,
).sort_index()

print(f"Rows: {len(data):,}")
print(f"Period: {data.index.min()} to {data.index.max()}")
print(f"Duplicate timestamps: {data.index.duplicated().sum()}")
print(f"Missing values: {int(data.isna().sum().sum())}")
display(data.head())

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 7))
window = data.iloc[: 7 * 48]
axes[0].plot(window["indoor_temperature_c"], label="Indoor")
axes[0].plot(window["outdoor_temperature_c"], label="Outdoor")
axes[0].set_ylabel("Temperature [°C]")
axes[0].legend()
axes[1].step(window.index, window["delivered_heat_kw"], where="post")
axes[1].set_ylabel("Useful heat [kW]")
axes[2].plot(window["solar_irradiance_w_m2"])
axes[2].set_ylabel("Irradiance [W/m²]")
axes[2].set_xlabel("Time")
plt.show()

## 2. Task 1 — aligned regression data

We fit the difference form

$$
T_{k+1}-T_k=b(T_{o,k}-T_k)+cq_{th,k}+\varepsilon_k
$$

where $b=\Delta t/(RC)$ and $c=\Delta t/C$. Inputs on row $k$ must be paired with the transition from $T_k$ to $T_{k+1}$.

In [ ]:
def make_regression_table(data):
    """Create aligned predictors at interval k and target T_(k+1)-T_k."""
    table = data.copy()
    # TODO 1.1: create temperature_change_c using the next indoor temperature.
    # TODO 1.2: create outdoor minus indoor temperature_difference_c.
    # TODO 1.3: remove the final row, whose next temperature is unavailable.
    raise NotImplementedError("Construct the aligned regression table.")

In [ ]:
regression_table = make_regression_table(data)
split_time = data.index.min() + pd.Timedelta(days=21)
calibration = regression_table.loc[regression_table.index < split_time]
validation_data = data.loc[data.index >= split_time]

print(f"Calibration rows: {len(calibration):,}")
print(f"Validation rows: {len(validation_data):,}")

## 3. Unconstrained worked example

In [ ]:
design_calibration = calibration[["temperature_difference_c", "delivered_heat_kw"]].to_numpy()
target_calibration = calibration["temperature_change_c"].to_numpy()
unconstrained_coefficients, *_ = np.linalg.lstsq(
    design_calibration, target_calibration, rcond=None
)
print("Unconstrained coefficients:", unconstrained_coefficients)

## 4. Task 2 — bounded estimation

Positive bounds encode the expected direction of outdoor heat exchange and heating. Bounds do not guarantee that the model is adequate; they prevent clearly impossible coefficient signs.

In [ ]:
def fit_bounded_rc(regression_table):
    """Return positive loss and heating coefficients from bounded least squares."""
    # TODO 2.1: build the two-column design matrix.
    # TODO 2.2: call lsq_linear with positive, physically meaningful bounds.
    # TODO 2.3: check result.success and return result.x.
    raise NotImplementedError("Fit the bounded RC coefficients.")

In [ ]:
loss_coefficient, heat_coefficient = fit_bounded_rc(calibration)
print(f"Loss coefficient b: {loss_coefficient:.5f}")
print(f"Heating coefficient c: {heat_coefficient:.5f} °C/kW per step")

## 5. Task 3 — recover effective parameters

$$
C=\frac{\Delta t}{c},\qquad R=\frac{c}{b},\qquad \tau=RC
$$

These are effective parameters for the selected model, sampling interval and operating data. They are not individual material properties.

In [ ]:
def recover_rc_parameters(loss_coefficient, heat_coefficient, step_hours):
    """Recover effective R, C and tau from the fitted difference equation."""
    # TODO 3: implement the three expressions given in the Markdown above.
    raise NotImplementedError("Recover the effective physical parameters.")

In [ ]:
estimated_r, estimated_c, estimated_tau = recover_rc_parameters(
    loss_coefficient, heat_coefficient, step_hours=0.5
)
print(f"R = {estimated_r:.3f} K/kW")
print(f"C = {estimated_c:.3f} kWh/K")
print(f"tau = {estimated_tau:.1f} h")

## 6. Task 4 — one-step and rollout validation

A one-step prediction uses the measured $T_k$ as the starting point for every transition. A **free rollout** starts from one measured temperature and then uses each predicted temperature as the starting point for the next transition. It receives no later indoor-temperature measurements, so small model errors can accumulate over the validation period.

In [ ]:
def one_step_predictions(data, loss_coefficient, heat_coefficient):
    """Reset from each measured temperature and predict the next value."""
    # TODO 4.1: implement the vectorised one-step prediction.
    raise NotImplementedError("Create one-step predictions.")


def rollout_predictions(data, loss_coefficient, heat_coefficient):
    """Initialise once and feed each prediction into the next step."""
    # TODO 4.2: implement the recursive rollout.
    raise NotImplementedError("Create rollout predictions.")

In [ ]:
def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if actual.shape != predicted.shape:
        raise ValueError("Actual and predicted values must have the same shape.")
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


one_step = one_step_predictions(validation_data, loss_coefficient, heat_coefficient)
rollout = rollout_predictions(validation_data, loss_coefficient, heat_coefficient)
actual_one_step = validation_data["indoor_temperature_c"].iloc[1:]
actual_rollout = validation_data["indoor_temperature_c"]

metrics = pd.DataFrame(
    {
        "RMSE [°C]": [
            rmse(actual_one_step, one_step),
            rmse(actual_rollout, rollout),
        ],
        "Bias [°C]": [
            float((one_step - actual_one_step).mean()),
            float((rollout - actual_rollout).mean()),
        ],
    },
    index=["One-step", "Free rollout"],
)
display(metrics.round(3))

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 7))
axes[0].plot(actual_rollout, label="Measured", color="black")
axes[0].plot(one_step, label="One-step", alpha=0.8)
axes[0].plot(rollout, label="Free rollout", alpha=0.8)
axes[0].set_ylabel("Temperature [°C]")
axes[0].legend()
rollout_residual = actual_rollout - rollout
axes[1].plot(rollout_residual)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Measured − rollout [°C]")
axes[1].set_xlabel("Time")
plt.show()

## 7. Residual diagnosis and supplied controller model

In [ ]:
residual_frame = pd.DataFrame(
    {
        "residual_c": rollout_residual,
        "heating_kw": validation_data["delivered_heat_kw"],
        "hour": validation_data.index.hour + validation_data.index.minute / 60,
    }
).dropna()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(residual_frame["hour"], residual_frame["residual_c"], s=8, alpha=0.5)
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Rollout residual [°C]")
axes[1].scatter(residual_frame["heating_kw"], residual_frame["residual_c"], s=8, alpha=0.5)
axes[1].set_xlabel("Delivered heat [kW]")
axes[1].set_ylabel("Rollout residual [°C]")
plt.tight_layout()
plt.show()

verified_model = json.loads(
    (ROOT / "data" / "reference_building" / "verified_controller_model.json").read_text()
)
display(pd.Series(verified_model, name="Verified model"))

## 8. Optional extension — occupied-house measurements

This extension applies the same identification workflow to measurements from an occupied semi-detached house in the UK. The two files contain five days at 10-minute resolution:

- November 2016, used for calibration;
- June 2017, used for independent validation without refitting.

The available `P_tot (W)` signal is a proxy for heat entering the building, not a direct measurement of useful space-heating output. The solar coefficient also absorbs glazing, shading and other effects. Treat the resulting coefficients and recovered $R$ and $C$ as effective parameters for this data and model.

Source: Hollick, F. and Wingfield, J. (2018), *Two periods of in-situ measurements from an occupied, semi-detached house in the UK*. https://doi.org/10.14324/000.ds.10087216

In [ ]:
measurement_directory = ROOT / "data" / "tutorial_6_2"


def load_house_period(filename):
    period = pd.read_csv(measurement_directory / filename, index_col=0)
    period.index = pd.to_datetime(period.index, format="%d/%m/%Y %H:%M")
    return period.sort_index()


winter_raw = load_house_period("10mins_solpap_2016.csv")
summer_raw = load_house_period("10mins_solpap_2017.csv")

measurement_summary = pd.DataFrame(
    {
        "Rows": [len(winter_raw), len(summer_raw)],
        "Start": [winter_raw.index.min(), summer_raw.index.min()],
        "End": [winter_raw.index.max(), summer_raw.index.max()],
        "Missing selected values": [
            int(winter_raw[["T_Average (degC)", "T_External (degC)", "P_tot (W)", "Solar (W/m2)"]].isna().sum().sum()),
            int(summer_raw[["T_Average (degC)", "T_External (degC)", "P_tot (W)", "Solar (W/m2)"]].isna().sum().sum()),
        ],
    },
    index=["November 2016", "June 2017"],
)
display(measurement_summary)

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 7))
axes[0].plot(winter_raw["T_Average (degC)"], label="Indoor average")
axes[0].plot(winter_raw["T_External (degC)"], label="Outdoor")
axes[0].set_ylabel("Temperature [°C]")
axes[0].legend()
axes[1].plot(winter_raw["P_tot (W)"] / 1000)
axes[1].set_ylabel("Power proxy [kW]")
axes[2].plot(winter_raw["Solar (W/m2)"])
axes[2].set_ylabel("Solar [W/m²]")
axes[2].set_xlabel("Time")
plt.show()

### Extension Task E1 — prepare the measured data

Interpolate the short measurement gaps separately within each five-day period. This is retrospective model identification, so interpolation across an internal gap is available. Then form

$$
T_{k+1}-T_k=b(T_{o,k}-T_k)+cP^{proxy}_k+gS_k+\varepsilon_k.
$$

Use `P_tot (W)` in kW and scale `Solar (W/m2)` to kW/m² so the fitted coefficients remain numerically convenient.

In [ ]:
def prepare_house_measurements(raw_data):
    """Prepare aligned indoor, outdoor, power and solar measurement columns."""
    # TODO E1.1: select the four columns named in the Markdown above.
    # TODO E1.2: interpolate the short gaps separately within this measurement period.
    # TODO E1.3: convert power and solar irradiance from W to kW-based units.
    # TODO E1.4: create the next temperature, temperature change and temperature difference.
    # TODO E1.5: remove the final row, whose next temperature is unavailable.
    raise NotImplementedError("Prepare the occupied-house measurement table.")

In [ ]:
winter_measurements = prepare_house_measurements(winter_raw)
summer_measurements = prepare_house_measurements(summer_raw)
display(winter_measurements.head(3))

### Extension Task E2 — fit on November 2016

Fit the three coefficients using the winter period only. Do not use June 2017 to choose coefficients or bounds.

In [ ]:
def fit_house_measurement_model(measurement_table):
    """Fit positive temperature-loss, power-proxy and solar coefficients."""
    # TODO E2.1: construct the three-column design matrix and target vector.
    # TODO E2.2: fit bounded coefficients with lsq_linear.
    # TODO E2.3: check result.success and return result.x.
    raise NotImplementedError("Fit the occupied-house measurement model.")

In [ ]:
house_coefficients = fit_house_measurement_model(winter_measurements)
house_loss_coefficient, house_power_coefficient, house_solar_coefficient = house_coefficients

house_step_hours = 1 / 6
house_effective_r = house_power_coefficient / house_loss_coefficient
house_effective_c = house_step_hours / house_power_coefficient
house_effective_tau = house_step_hours / house_loss_coefficient

print(f"Loss coefficient: {house_loss_coefficient:.6f}")
print(f"Power-proxy coefficient: {house_power_coefficient:.6f}")
print(f"Solar coefficient: {house_solar_coefficient:.6f}")
print(f"Effective R under the proxy assumption: {house_effective_r:.2f} K/kW")
print(f"Effective C under the proxy assumption: {house_effective_c:.2f} kWh/K")
print(f"Effective time constant: {house_effective_tau:.1f} h")

### Extension Task E3 — test without refitting

Compare one-step prediction and a free rollout on both periods. The 2017 period is a demanding test because it has warmer weather and much less gas use than the calibration period.

In [ ]:
def predict_house_measurements(measurement_table, coefficients):
    """Return measured, one-step and free-rollout temperature series."""
    # TODO E3.1: calculate one-step predictions using measured T_k each time.
    # TODO E3.2: calculate a free rollout using only the first measured temperature.
    # TODO E3.3: return aligned actual, one-step and rollout Series.
    raise NotImplementedError("Evaluate the occupied-house measurement model.")

In [ ]:
house_results = {}
house_metrics = {}
for period_name, period_table in {
    "2016 calibration": winter_measurements,
    "2017 validation": summer_measurements,
}.items():
    actual, one_step_house, rollout_house = predict_house_measurements(
        period_table, house_coefficients
    )
    house_results[period_name] = (actual, one_step_house, rollout_house)
    house_metrics[period_name] = {
        "One-step RMSE [°C]": rmse(actual, one_step_house),
        "Free-rollout RMSE [°C]": rmse(actual, rollout_house),
        "Free-rollout bias [°C]": float((rollout_house - actual).mean()),
    }

house_metric_table = pd.DataFrame(house_metrics).T
display(house_metric_table.round(3))

actual, one_step_house, rollout_house = house_results["2017 validation"]
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 7))
axes[0].plot(actual, label="Measured", color="black")
axes[0].plot(one_step_house, label="One-step", alpha=0.8)
axes[0].plot(rollout_house, label="Free rollout", alpha=0.8)
axes[0].set_ylabel("Temperature [°C]")
axes[0].legend()
axes[1].plot(actual - one_step_house, label="One-step residual")
axes[1].plot(actual - rollout_house, label="Free-rollout residual", alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Measured − predicted [°C]")
axes[1].set_xlabel("Time")
axes[1].legend()
plt.show()

### Extension questions

1. Why can the one-step RMSE remain small while the free rollout drifts substantially?
2. How do the June 2017 operating conditions differ from the November 2016 calibration period?
3. Why should coefficients based on `P_tot (W)` be treated as effective rather than physical parameters?
4. Which additional measurements would make the identification problem more physically interpretable?

## Discussion

1. Why is the rollout error larger than the one-step error?
2. Which omitted effects could explain a residual pattern by hour of day?
3. Why will later tutorials use the common verified model rather than every student's fitted model?
4. Over what horizon and operating range does the fitted model appear credible?